# Notebook 12: Lua Scripting — Custom Atomic Operations

Sometimes Redis's built-in commands aren't enough. You need custom logic that runs **atomically** (no other commands can interrupt it). That's where **Lua scripting** comes in.

Lua scripts run **directly on the Redis server** as a single atomic operation. While a script is running, nothing else can happen — it's like a super-powered transaction.

**When to use Lua:**
- Read-then-write logic that must be atomic
- Complex operations that MULTI/EXEC can't handle
- Reducing round trips for multi-step operations

In [ ]:
import redis
import time

r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
r.flushdb()
print("Connected and ready!")

---
## 1. Basic EVAL — Hello Lua!

`EVAL script numkeys [key ...] [arg ...]`
- `script`: Lua code as a string
- `numkeys`: How many of the following arguments are key names
- `KEYS[]`: Lua array of key names (1-indexed!)
- `ARGV[]`: Lua array of additional arguments

In [ ]:
# Simplest Lua script — just return a value
# Redis CLI: EVAL "return 'Hello from Lua!'" 0
result = r.eval("return 'Hello from Lua!'", 0)
print(f"Result: {result}")

# Return a number
result = r.eval("return 42", 0)
print(f"Number: {result}")

# Math in Lua
result = r.eval("return 10 + 20 + 30", 0)
print(f"Math: {result}")

## 2. Lua + Redis Commands

In [ ]:
# Call Redis commands from Lua using redis.call()
r.set('greeting', 'Hello from Redis!')

# Redis CLI: EVAL "return redis.call('GET', KEYS[1])" 1 greeting
result = r.eval("return redis.call('GET', KEYS[1])", 1, 'greeting')
print(f"GET via Lua: {result}")

# SET via Lua
r.eval("redis.call('SET', KEYS[1], ARGV[1])", 1, 'lua_key', 'lua_value')
print(f"SET via Lua: {r.get('lua_key')}")

# KEYS[1] = first key argument
# ARGV[1] = first non-key argument
# Note: Lua arrays are 1-indexed (not 0-indexed like Python!)

## 3. Variables and Logic

In [ ]:
# Conditional logic in Lua
r.set('status', 'active')

script = """
local current = redis.call('GET', KEYS[1])
if current == ARGV[1] then
    return 'Match! Status is: ' .. current
else
    return 'No match. Status is: ' .. tostring(current) .. ', expected: ' .. ARGV[1]
end
"""

result = r.eval(script, 1, 'status', 'active')
print(f"Check 'active': {result}")

result = r.eval(script, 1, 'status', 'inactive')
print(f"Check 'inactive': {result}")

In [ ]:
# Using variables and loops
script = """
local count = 0
for i = 1, #ARGV do
    redis.call('SET', KEYS[1] .. ':' .. i, ARGV[i])
    count = count + 1
end
return count
"""

# Set 3 keys using a single Lua script
result = r.eval(script, 1, 'item', 'apple', 'banana', 'cherry')
print(f"Set {result} keys")
print(f"  item:1 = {r.get('item:1')}")
print(f"  item:2 = {r.get('item:2')}")
print(f"  item:3 = {r.get('item:3')}")

## 4. redis.call() vs redis.pcall()

- `redis.call()` — Stops the script if a command fails (error propagates)
- `redis.pcall()` — Catches errors, returns an error object (script continues)

In [ ]:
# redis.pcall example — handle errors gracefully
r.set('mystring', 'hello')  # This is a string, not a list

script = """
local ok, err = pcall(redis.call, 'LPUSH', KEYS[1], 'value')
if not ok then
    return 'Error caught: ' .. tostring(err)
else
    return 'Success'
end
"""

result = r.eval(script, 1, 'mystring')
print(f"Result: {result}")

## 5. Script Caching with EVALSHA and register_script

In [ ]:
# Method 1: EVALSHA — Cache the script, send only its hash
script_text = "return redis.call('GET', KEYS[1])"

# Load script and get SHA1 hash
sha = r.script_load(script_text)
print(f"Script SHA1: {sha}")

# Now use EVALSHA — sends just the hash, not the whole script
r.set('test', 'hello')
result = r.evalsha(sha, 1, 'test')
print(f"EVALSHA result: {result}")

In [ ]:
# Method 2: register_script — The Pythonic way (recommended!)
# It automatically handles EVALSHA and fallback to EVAL

# Define the script
get_and_increment = r.register_script("""
local current = redis.call('GET', KEYS[1])
if current then
    redis.call('INCR', KEYS[1])
    return current
else
    redis.call('SET', KEYS[1], 1)
    return '0'
end
""")

# Use it like a function!
r.delete('counter')
for i in range(5):
    old_value = get_and_increment(keys=['counter'])
    print(f"  Call {i+1}: old value was {old_value}, new value is {r.get('counter')}")

---
## 6. Real-World: Atomic Rate Limiter

In [ ]:
# Rate limiter as a single atomic Lua script
# This is MUCH safer than the Python version (no race conditions!)

rate_limit_script = r.register_script("""
local key = KEYS[1]
local max_requests = tonumber(ARGV[1])
local window = tonumber(ARGV[2])

local current = tonumber(redis.call('GET', key) or '0')

if current >= max_requests then
    local ttl = redis.call('TTL', key)
    return {0, ttl}  -- Blocked, return remaining time
end

current = redis.call('INCR', key)
if current == 1 then
    redis.call('EXPIRE', key, window)
end

return {1, max_requests - current}  -- Allowed, return remaining quota
""")

r.delete('ratelimit:user1')

for i in range(8):
    result = rate_limit_script(keys=['ratelimit:user1'], args=[5, 60])
    allowed, remaining = result
    status = 'ALLOWED' if allowed else 'BLOCKED'
    print(f"  Request {i+1}: [{status}] remaining={remaining}")

---
## 7. Real-World: Compare-and-Swap (CAS)

In [ ]:
# Atomic compare-and-swap: update only if current value matches expected
cas_script = r.register_script("""
local current = redis.call('GET', KEYS[1])
if current == ARGV[1] then
    redis.call('SET', KEYS[1], ARGV[2])
    return 1  -- Success
else
    return 0  -- Failed (value changed)
end
""")

r.set('config:version', 'v1')

# Try to update v1 → v2
result = cas_script(keys=['config:version'], args=['v1', 'v2'])
print(f"CAS v1→v2: {'Success' if result else 'Failed'}")
print(f"Value: {r.get('config:version')}")

# Try to update v1 → v3 (should fail because it's now v2)
result = cas_script(keys=['config:version'], args=['v1', 'v3'])
print(f"CAS v1→v3: {'Success' if result else 'Failed (expected v1, found v2)'}")
print(f"Value: {r.get('config:version')}")

---
## 8. Real-World: Conditional Update

In [ ]:
# Transfer money only if sender has enough balance (fully atomic)
transfer_script = r.register_script("""
local from_balance = tonumber(redis.call('GET', KEYS[1]))
local amount = tonumber(ARGV[1])

if from_balance == nil then
    return {0, 'Source account not found'}
end

if from_balance < amount then
    return {0, 'Insufficient funds: ' .. tostring(from_balance)}
end

redis.call('DECRBY', KEYS[1], amount)
redis.call('INCRBY', KEYS[2], amount)
return {1, 'Transferred ' .. tostring(amount)}
""")

r.set('balance:alice', 1000)
r.set('balance:bob', 500)

# Transfer $300 from Alice to Bob
result = transfer_script(keys=['balance:alice', 'balance:bob'], args=[300])
print(f"Transfer $300: {result}")
print(f"Alice: ${r.get('balance:alice')}, Bob: ${r.get('balance:bob')}")

# Try to transfer more than Alice has
result = transfer_script(keys=['balance:alice', 'balance:bob'], args=[9999])
print(f"Transfer $9999: {result}")

---
## 9. Best Practices

| Do | Don't |
|---|---|
| Keep scripts short and fast | Run long-running scripts (blocks server!) |
| Use KEYS[] for all key access | Hardcode key names (breaks cluster) |
| Use register_script() in Python | Send full script text every time |
| Return meaningful values | Ignore error handling |
| Test scripts thoroughly | Use complex Lua libraries |

---
## Cleanup

In [ ]:
r.flushdb()
print("Cleaned up!")

---
## Key Takeaways

```python
# Basic eval
r.eval("return redis.call('GET', KEYS[1])", 1, 'mykey')

# Script caching
sha = r.script_load(script_text)
r.evalsha(sha, 1, 'mykey')

# Pythonic way (recommended)
my_script = r.register_script("""lua code here""")
result = my_script(keys=['key1', 'key2'], args=['arg1', 'arg2'])
```

### Lua Quick Reference
```lua
local x = 10                    -- variable
redis.call('SET', KEYS[1], x)   -- call Redis command
if x > 5 then ... end           -- conditional
for i = 1, 10 do ... end        -- loop
return {1, 'ok'}                -- return array
tostring(x) / tonumber(s)       -- type conversion
```

---
## Exercises

1. **Atomic Get-or-Set:** Write a Lua script that gets a key's value if it exists, or sets it to a default and returns the default.

2. **Sliding Window Rate Limiter:** Write a Lua script that implements a sliding window counter using a sorted set (ZADD with timestamps, ZREMRANGEBYSCORE to clean old entries, ZCARD to count).

3. **Atomic Swap:** Write a Lua script that swaps the values of two keys atomically.

4. **Conditional Batch Update:** Write a Lua script that increments multiple counters only if ALL of them are below a maximum value. If any would exceed the max, none should be incremented.

In [ ]:
# Your exercises here!


---
**Next up: [Notebook 13 — Cluster & Sentinel](./13_Cluster_and_Sentinel.ipynb)** — High availability, replication, and scaling Redis for production!